In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. TẢI DỮ LIỆU (Load Data)
# ==========================================
# Đảm bảo các file csv nằm cùng thư mục với script
df_info = pd.read_csv('../data/oulad/studentInfo.csv')
df_stu_assess = pd.read_csv('../data/oulad/studentAssessment.csv')
df_assess = pd.read_csv('../data/oulad/assessments.csv')
df_vle = pd.read_csv('../data/oulad/studentVle.csv')
df_vle_info = pd.read_csv('../data/oulad/vle.csv')

# Thiết lập mốc thời gian cảnh báo sớm (Ví dụ: Tuần 8 ~ Ngày 60 của khóa học)
CUTOFF_DAY = 60

# ==========================================
# 2. XỬ LÝ NHÃN (Target) & THÔNG TIN CƠ SỞ
# ==========================================
# Gộp chung Fail và Withdrawn thành nhóm "Có rủi ro" (Risk = 1)
df_info['is_Risk'] = df_info['final_result'].map({
    'Pass': 0, 'Distinction': 0, 
    'Fail': 1, 'Withdrawn': 1
})

# Lấy các trường dữ liệu nền tảng làm bộ khung (Base DataFrame)
df_base = df_info[['id_student', 'code_module', 'code_presentation', 
                   'num_of_prev_attempts', 'studied_credits', 'is_Risk']].copy()


# ==========================================
# 3. TRÍCH XUẤT ĐẶC TRƯNG ĐIỂM SỐ (Assessments)
# ==========================================
# Gộp bảng điểm của sinh viên với thông tin bài kiểm tra
df_ass_merged = pd.merge(df_stu_assess, df_assess, on='id_assessment', how='left')

# ĐIỀU KIỆN LỌC QUAN TRỌNG:
# 1. Bỏ qua bài thi cuối kỳ (Exam)
# 2. Chỉ tính các bài tập có hạn nộp (date) <= CUTOFF_DAY
df_ass_filtered = df_ass_merged[
    (df_ass_merged['assessment_type'] != 'Exam') & 
    (df_ass_merged['date'] <= CUTOFF_DAY)
].copy()

# Tính số ngày nộp trễ (Nếu nộp sớm, gán = 0)
df_ass_filtered['late_days'] = df_ass_filtered['date_submitted'] - df_ass_filtered['date']
df_ass_filtered['late_days'] = df_ass_filtered['late_days'].apply(lambda x: x if x > 0 else 0)

# Gom nhóm (Groupby) theo từng sinh viên & môn học
ass_features = df_ass_filtered.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    Avg_Coursework_Score=('score', 'mean'),       # Điểm trung bình quá trình
    Total_late_days=('late_days', 'sum'),         # Tổng số ngày nộp trễ
    Submitted_assignments=('id_assessment', 'count') # Số bài đã nộp tính đến tuần 8
).reset_index()

df_first_ass = df_ass_filtered.sort_values(['id_student', 'code_module', 'code_presentation', 'date'])
df_first_ass = df_first_ass.drop_duplicates(subset=['id_student', 'code_module', 'code_presentation'], keep='first')
df_first_ass = df_first_ass[['id_student', 'code_module', 'code_presentation', 'score']].rename(columns={'score': 'First_Assessment_Score'})


# ==========================================
# 4. TRÍCH XUẤT ĐẶC TRƯNG TƯƠNG TÁC (Clickstream/LMS)
# ==========================================
# LỌC QUAN TRỌNG: Chỉ lấy lịch sử click <= CUTOFF_DAY
df_vle_filtered = df_vle[df_vle['date'] <= CUTOFF_DAY]

# Gom nhóm (Groupby) tính tổng số click và số ngày truy cập
vle_features = df_vle_filtered.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    Total_clicks=('sum_click', 'sum'),            # Tổng số lượt click
    Active_days=('date', 'nunique')               # Số ngày khác nhau có đăng nhập BKeL
).reset_index()

# [MỚI] Ghép với vle_info để lấy loại hoạt động (activity_type)
df_vle_merged = pd.merge(df_vle_filtered, df_vle_info, on=['id_site', 'code_module', 'code_presentation'], how='left')

# [MỚI] Lọc các click thuộc diễn đàn (Forum_clicks)
df_forum = df_vle_merged[df_vle_merged['activity_type'] == 'forumng']
forum_features = df_forum.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    Forum_clicks=('sum_click', 'sum')
).reset_index()

# [MỚI] Tính động lượng (Click_Momentum)
# Chia làm 2 giai đoạn: Giai đoạn 1 (từ đầu đến ngày 30), Giai đoạn 2 (ngày 31 đến 60)
df_vle_filtered['Phase'] = np.where(df_vle_filtered['date'] <= 30, 'Phase1', 'Phase2')
phase_clicks = df_vle_filtered.groupby(['id_student', 'code_module', 'code_presentation', 'Phase'])['sum_click'].sum().unstack(fill_value=0).reset_index()

# Đảm bảo có đủ 2 cột dù có thể sinh viên không tương tác
if 'Phase1' not in phase_clicks.columns: phase_clicks['Phase1'] = 0
if 'Phase2' not in phase_clicks.columns: phase_clicks['Phase2'] = 0

# Công thức: Click Phase 2 / (Click Phase 1 + 1) để tránh lỗi chia cho 0
phase_clicks['Click_Momentum'] = (phase_clicks['Phase2'] / (phase_clicks['Phase1'] + 1)).round(2)
momentum_features = phase_clicks[['id_student', 'code_module', 'code_presentation', 'Click_Momentum']]

df_vle_filtered = df_vle[df_vle['date'] <= CUTOFF_DAY].copy()

# ==========================================
# 5. GỘP TOÀN BỘ (Merge & Fill Missing)
# ==========================================
# Dùng LEFT JOIN để giữ lại mọi sinh viên (kể cả người chưa nộp bài/chưa click)
df_final = df_base.merge(ass_features, on=['id_student', 'code_module', 'code_presentation'], how='left')
df_final = df_final.merge(df_first_ass, on=['id_student', 'code_module', 'code_presentation'], how='left')
df_final = df_final.merge(vle_features, on=['id_student', 'code_module', 'code_presentation'], how='left')
df_final = df_final.merge(forum_features, on=['id_student', 'code_module', 'code_presentation'], how='left')
df_final = df_final.merge(momentum_features, on=['id_student', 'code_module', 'code_presentation'], how='left')

# Xử lý các giá trị NaN sinh ra do Left Join
# Sinh viên không có dữ liệu nghĩa là điểm quá trình = 0, click = 0
df_final.fillna({
    'Avg_Coursework_Score': 0,
    'First_Assessment_Score': 0,
    'Total_late_days': 0,
    'Submitted_assignments': 0,
    'Total_clicks': 0,
    'Active_days': 0,
    'Forum_clicks': 0,           # [MỚI]
    'Click_Momentum': 0          # [MỚI]
}, inplace=True)


# XEM KẾT QUẢ
print("Kích thước dữ liệu cuối cùng:", df_final.shape)


df_final.to_csv('../data/processed/df_final_prepared.csv', index=False)

print("Đã xuất dữ liệu ra file 'df_final_prepared.csv' thành công!")



Kích thước dữ liệu cuối cùng: (32593, 14)
Đã xuất dữ liệu ra file 'df_final_prepared.csv' thành công!
   id_student code_module code_presentation  num_of_prev_attempts  \
0       11391         AAA             2013J                     0   

   studied_credits  is_Risk  Avg_Coursework_Score  Total_late_days  \
0              240        0                  81.5              0.0   

   Submitted_assignments  First_Assessment_Score  Total_clicks  Active_days  \
0                    2.0                    78.0         529.0         18.0   

   Forum_clicks  Click_Momentum  
0          98.0            0.25  


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# ==========================================
# 1. ĐỌC DỮ LIỆU TỪ FILE CSV
# ==========================================
# Đọc file dữ liệu đã tiền xử lý
df = pd.read_csv('../data/processed/df_final_prepared.csv')
print(f"Kích thước dữ liệu đọc vào: {df.shape}")

# ==========================================
# 2. TÁCH BIẾN X (FEATURES) VÀ y (TARGET)
# ==========================================
# Loại bỏ các cột định danh và nhãn để tạo tập X
X = df.drop(columns=['id_student', 'code_module', 'code_presentation', 'is_Risk'])

# Lấy riêng cột nhãn làm tập y
y = df['is_Risk']

# ==========================================
# 3. CHIA TẬP TRAIN / TEST
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y # Đảm bảo tỷ lệ rớt/đậu ở Train và Test bằng nhau
)
print(f"Số lượng mẫu Train: {len(X_train)} | Test: {len(X_test)}")

# ==========================================
# 4. HUẤN LUYỆN MÔ HÌNH XGBOOST
# ==========================================
xgb_model = XGBClassifier(
    n_estimators=100,      
    max_depth=5,           
    learning_rate=0.1,     
    random_state=42,       
    eval_metric='logloss'  
)

xgb_model.fit(X_train, y_train)
print("\nĐã huấn luyện xong mô hình XGBoost!")

# ==========================================
# 5. DỰ ĐOÁN VÀ ĐÁNH GIÁ (EVALUATION)
# ==========================================
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("\n--- BÁO CÁO ĐÁNH GIÁ MÔ HÌNH ---")
print(classification_report(y_test, y_pred, target_names=['An toàn (0)', 'Rủi ro (1)']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")


xgb_model.save_model('../models/xgboost_risk_model.json')
print("Đã lưu mô hình dạng JSON an toàn!")




Kích thước dữ liệu đọc vào: (32593, 14)
Số lượng mẫu Train: 26074 | Test: 6519

Đã huấn luyện xong mô hình XGBoost!

--- BÁO CÁO ĐÁNH GIÁ MÔ HÌNH ---
              precision    recall  f1-score   support

 An toàn (0)       0.76      0.86      0.81      3077
  Rủi ro (1)       0.86      0.76      0.81      3442

    accuracy                           0.81      6519
   macro avg       0.81      0.81      0.81      6519
weighted avg       0.82      0.81      0.81      6519

ROC-AUC Score: 0.8850
Đã lưu mô hình dạng JSON an toàn!
